# ConvoBridge — Gemma 3 Meeting Summarization (QLoRA)

**Updated for:** 2 epochs + more data + max GPU use on Colab T4 (~15 GB)

## Before start
1. Runtime → Change runtime type → **GPU (T4)**
2. Accept Gemma license: https://huggingface.co/google/gemma-3-1b-it
3. HF token: https://huggingface.co/settings/tokens

Checkpoints save to Google Drive so Colab crashes don't lose progress.

## Step 1 — Check GPU

In [ ]:
!nvidia-smi

## Step 2 — Install dependencies

In [ ]:
!pip install -q "transformers>=4.49.0" "peft>=0.13.0" "bitsandbytes>=0.44.0" \
  "datasets>=3.0.0" "accelerate>=1.0.0" "trl>=0.12.0" huggingface_hub pyyaml

## Step 3 — HF login + mount Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
Path("/content/drive/MyDrive/ConvoBridge").mkdir(parents=True, exist_ok=True)
Path("/content/drive/MyDrive/ConvoBridge/checkpoints").mkdir(parents=True, exist_ok=True)
print("Google Drive mounted.")

login()
# login(token="hf_xxxxxxxx")

## Step 4 — Configuration (2 epochs + more data + max GPU)

In [ ]:
from pathlib import Path

# Same base model as backend SUMMARIZATION_MODEL
BASE_MODEL = "google/gemma-3-1b-it"

DRIVE_ROOT = Path("/content/drive/MyDrive/ConvoBridge")
OUTPUT_DIR = DRIVE_ROOT / "gemma3-meeting-lora"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"

DATASET_SOURCES = [
    {"id": "knkarthick/samsum", "type": "dialogue"},
    {"id": "EdinburghNLP/xsum", "type": "document", "split": "train", "max_rows": 4000},
]

TRAIN_CFG = {
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "learning_rate": 1e-4,
    "num_epochs": 2,                 # requested
    "per_device_batch_size": 24,     # push T4 VRAM hard (drop to 16/12 if OOM)
    "gradient_accumulation_steps": 1,
    "max_seq_length": 1536,          # longer context uses more GPU
    "save_steps": 100,
    "max_train_rows": 8000,          # more data
}

PROMPT_TEMPLATE = """You are ConvoBridge Meeting Assistant. Summarize the meeting transcript below.

Return ONLY the following sections (no extra text):

SUMMARY:
<2-4 sentence overview>

KEY_POINTS:
- <bullet 1>
- <bullet 2>
- <bullet 3>

ACTION_ITEMS:
- <owner or team>: <task> (due: <if mentioned, else TBD>)
- <owner or team>: <task> (due: TBD)

Rules:
- Use only information from the transcript.
- If no action items exist, write: ACTION_ITEMS:\n- None
- Keep language same as transcript unless user asks otherwise.

TRANSCRIPT:
{transcript}"""

eff_batch = TRAIN_CFG["per_device_batch_size"] * TRAIN_CFG["gradient_accumulation_steps"]
est_steps = (TRAIN_CFG["max_train_rows"] // eff_batch) * TRAIN_CFG["num_epochs"]
print("Model:", BASE_MODEL)
print("Epochs:", TRAIN_CFG["num_epochs"], "| Rows:", TRAIN_CFG["max_train_rows"])
print("Batch:", TRAIN_CFG["per_device_batch_size"], "| Seq:", TRAIN_CFG["max_seq_length"])
print("Estimated steps ~", est_steps)
print("If CUDA OOM: set per_device_batch_size to 16 then 12 then 8")

## Step 5 — Load online datasets (Hugging Face)

In [ ]:
import re
from datasets import load_dataset


def extract_key_sentences(text: str, max_points: int = 5) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 10][:max_points]


def wrap_summary(summary: str) -> str:
    summary = summary.strip()
    key_points = extract_key_sentences(summary)
    kp_block = "\n".join(f"- {kp}" for kp in key_points) if key_points else f"- {summary}"
    return (
        "SUMMARY:\n"
        f"{summary}\n\n"
        "KEY_POINTS:\n"
        f"{kp_block}\n\n"
        "ACTION_ITEMS:\n"
        "- None"
    )


def row_from_samsum(ex: dict):
    text_in = (ex.get("dialogue") or "").strip()
    text_out = (ex.get("summary") or "").strip()
    if not text_in or not text_out:
        return None
    return {"input": text_in, "output": wrap_summary(text_out), "source": "samsum"}


def row_from_document(ex: dict):
    text_in = (ex.get("document") or ex.get("article") or "").strip()
    text_out = (ex.get("summary") or ex.get("highlights") or "").strip()
    if not text_in or not text_out:
        return None
    return {"input": text_in, "output": wrap_summary(text_out), "source": "document"}


def load_online_rows() -> list[dict]:
    rows: list[dict] = []
    max_rows = TRAIN_CFG["max_train_rows"]

    for src in DATASET_SOURCES:
        ds_id = src["id"]
        print(f"Loading from Hugging Face: {ds_id}")
        try:
            ds = load_dataset(ds_id)
        except Exception as exc:
            print(f"  Skipped {ds_id}: {exc}")
            continue

        split_name = src.get("split") or ("train" if "train" in ds else list(ds.keys())[0])
        split = ds[split_name]
        cap = src.get("max_rows")
        if cap:
            split = split.select(range(min(cap, len(split))))

        parser = row_from_samsum if src["type"] == "dialogue" else row_from_document
        added = 0
        for ex in split:
            row = parser(ex)
            if row:
                rows.append(row)
                added += 1
            if len(rows) >= max_rows:
                break
        print(f"  Added {added} rows from {ds_id} ({split_name})")
        if len(rows) >= max_rows:
            break

    print(f"Total training rows: {len(rows)}")
    return rows


train_rows = load_online_rows()
assert len(train_rows) > 0, "No training data loaded."
train_rows[0]

## Step 6 — Format Gemma chat examples

In [ ]:
from datasets import Dataset


def format_example(row: dict) -> str:
    user_content = PROMPT_TEMPLATE.format(transcript=row["input"])
    return (
        f"<start_of_turn>user\n{user_content}<end_of_turn>\n"
        f"<start_of_turn>model\n{row['output']}<end_of_turn>"
    )


formatted = [format_example(r) for r in train_rows]
train_ds = Dataset.from_dict({"text": formatted})
print("Rows:", len(train_ds))
print(formatted[0][:900], "...")

## Step 7 — Train (max GPU + resumable)

If CUDA OOM: change `per_device_batch_size` in Step 4 to **16 → 12 → 8** and re-run this cell.

If Colab disconnects: re-run Steps 1–7 — it resumes from Drive checkpoint.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_config = LoraConfig(
    r=TRAIN_CFG["lora_r"],
    lora_alpha=TRAIN_CFG["lora_alpha"],
    lora_dropout=TRAIN_CFG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

resume_ckpt = None
ckpts = sorted(
    CHECKPOINT_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)
if ckpts:
    resume_ckpt = str(ckpts[-1])
    print("Resuming from:", resume_ckpt)
else:
    print("No checkpoint found — starting fresh.")


def build_trainer(model, tokenizer, train_ds):
    common = dict(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=TRAIN_CFG["num_epochs"],
        per_device_train_batch_size=TRAIN_CFG["per_device_batch_size"],
        gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
        learning_rate=TRAIN_CFG["learning_rate"],
        logging_steps=10,
        save_steps=TRAIN_CFG["save_steps"],
        save_total_limit=3,
        bf16=torch.cuda.is_available(),
        # Keep gradient_checkpointing OFF so GPU stays busy / higher util
        gradient_checkpointing=False,
        optim="paged_adamw_8bit",
        report_to="none",
        dataloader_pin_memory=True,
        dataloader_num_workers=2,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
    )

    if SFTConfig is not None:
        try:
            args = SFTConfig(max_length=TRAIN_CFG["max_seq_length"], **common)
            return SFTTrainer(
                model=model,
                args=args,
                train_dataset=train_ds,
                processing_class=tokenizer,
            )
        except TypeError:
            pass

    args = TrainingArguments(**common)
    try:
        return SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            processing_class=tokenizer,
            max_seq_length=TRAIN_CFG["max_seq_length"],
        )
    except TypeError:
        return SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            tokenizer=tokenizer,
            max_seq_length=TRAIN_CFG["max_seq_length"],
        )


trainer = build_trainer(model, tokenizer, train_ds)

print("GPU allocated before train:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
print("GPU reserved before train:", round(torch.cuda.memory_reserved() / 1e9, 2), "GB")

trainer.train(resume_from_checkpoint=resume_ckpt)

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved LoRA adapter ->", OUTPUT_DIR)
print("GPU peak reserved:", round(torch.cuda.max_memory_reserved() / 1e9, 2), "GB")

## Step 8 — Inference test (fixed)

In [ ]:
model.eval()
model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

sample_transcript = """
Ahmed: We need to finish the project report by Friday.
Sara: I will handle the design section.
John: I can review the budget numbers tonight.
Ahmed: Great. Let's meet again on Monday at 10 AM.
""".strip()

prompt = (
    f"<start_of_turn>user\n{PROMPT_TEMPLATE.format(transcript=sample_transcript)}<end_of_turn>\n"
    f"<start_of_turn>model\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.inference_mode():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = out[0][inputs["input_ids"].shape[-1] :]
print(tokenizer.decode(generated, skip_special_tokens=True))

## Step 9 — Download zip to PC

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/gemma3-meeting-lora"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(zip_path + ".zip")
print("Unzip to: summarization/models/gemma3-meeting-lora/")

## Step 10 — Use in backend

1. Put adapter files in:
   `summarization/models/gemma3-meeting-lora/`
2. Need `adapter_config.json` + `adapter_model.safetensors`
3. Start API and test `POST /summarize`